# 03 · Silver · Olist (canal marketplace)
Limpa, tipa e aplica as regras de negócio aos dados do marketplace.
**Nenhuma linha é removida:** os itens recebem marcações (`valido`, `incluir`, `periodo_completo`) e o filtro acontece na gold. Isso preserva a rastreabilidade e permite medir o efeito de cada regra.

| Regra | Coluna | Origem da decisão |
|---|---|---|
| Pedido é venda real (não cancelado/indisponível, com itens) | `valido` | Perfil, consultas 2 e 5 |
| Produto pertence ao mix de moda da Horizonte | `incluir` | Perfil, consulta 1 + de-para |
| Data dentro da janela confiável (jan/2017–ago/2018) | `periodo_completo` | Perfil, consulta 4 |

No GCP, estas transformações rodariam no BigQuery via Dataform. A versão BigQuery fica em `sql/bigquery/`.

### 1. Parâmetros
`deslocamento_dias = 2926` (418 semanas): leva a Black Friday de 2017 para a de 2025, mantendo o dia da semana. O histórico do marketplace passa a ir de 05/01/2025 a 04/09/2026.

In [0]:
DECLARE OR REPLACE VARIABLE deslocamento_dias INT DEFAULT 2926;  -- 418 semanas: BF 2017 -> BF 2025

### 2. Tabela de referência do de-para
Carrega `de_para_categorias.csv`, mantido no repositório. A lista de categorias veio do perfil dos dados; o mapeamento é uma **decisão de negócio** do squad, documentada na coluna `justificativa`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.ref_de_para_categorias AS
SELECT categoria_olist,
       NULLIF(categoria_horizonte, '')  AS categoria_horizonte,
       CAST(incluir AS BOOLEAN)         AS incluir,
       justificativa
FROM read_files('/Volumes/horizonte/bronze/landing/referencia/de_para_categorias.csv',
                format => 'csv', header => true);

### 3. Pedidos
- Tipa a data da compra e aplica o deslocamento (guarda também a data original).
- Usa `customer_unique_id` como cliente: no Olist, `customer_id` é gerado por pedido.
- Marca `valido` e `periodo_completo`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.marketplace_pedidos AS
WITH itens_por_pedido AS (
  SELECT order_id, COUNT(*) AS qtd_itens
  FROM horizonte.bronze.olist_order_items
  GROUP BY order_id
)
SELECT o.order_id                                              AS pedido_id,
       c.customer_unique_id                                    AS cliente_id,
       c.customer_state                                        AS uf,
       o.order_status                                          AS status,
       CAST(o.order_purchase_timestamp AS TIMESTAMP)           AS data_compra_original,
       timestampadd(DAY, deslocamento_dias,
                    CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS data_compra,
       COALESCE(i.qtd_itens, 0)                                AS qtd_itens,
       o.order_status NOT IN ('canceled', 'unavailable')
         AND COALESCE(i.qtd_itens, 0) > 0                      AS valido,
       CAST(o.order_purchase_timestamp AS DATE)
         BETWEEN DATE'2017-01-01' AND DATE'2018-08-31'         AS periodo_completo
FROM horizonte.bronze.olist_orders o
LEFT JOIN horizonte.bronze.olist_customers c USING (customer_id)
LEFT JOIN itens_por_pedido i USING (order_id);

### 4. Itens
- Tipa preço e frete como `DECIMAL(12,2)`.
- Cada linha de `order_items` é uma unidade vendida (`quantidade = 1`).
- Associa a categoria da Horizonte pelo de-para; categorias ausentes ou nulas viram `Não classificado` com `incluir = false`.
- Herda `valido` e `periodo_completo` do pedido.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.marketplace_itens AS
SELECT i.order_id                                  AS pedido_id,
       CAST(i.order_item_id AS INT)                AS item_seq,
       i.product_id                                AS produto_id,
       p.data_compra,
       p.valido,
       p.periodo_completo,
       CAST(i.price AS DECIMAL(12,2))              AS preco,
       CAST(i.freight_value AS DECIMAL(12,2))      AS frete,
       1                                           AS quantidade,   -- cada linha do Olist é uma unidade
       pr.product_category_name                    AS categoria_olist,
       COALESCE(d.categoria_horizonte, 'Não classificado') AS categoria,
       COALESCE(d.incluir, FALSE)                  AS incluir
FROM horizonte.bronze.olist_order_items i
JOIN horizonte.silver.marketplace_pedidos p ON p.pedido_id = i.order_id
LEFT JOIN horizonte.bronze.olist_products pr ON pr.product_id = i.product_id
LEFT JOIN horizonte.silver.ref_de_para_categorias d
       ON d.categoria_olist = pr.product_category_name;

### 5. Conferência
Valores esperados: 73 categorias · 99.441 pedidos · 98.199 válidos · 112.650 itens · **42.758 itens para a gold** · fim da janela em 04/09/2026.

In [0]:
SELECT
  (SELECT COUNT(*) FROM horizonte.silver.ref_de_para_categorias)               AS categorias_de_para, -- esperado 73
  (SELECT COUNT(*) FROM horizonte.silver.marketplace_pedidos)                  AS pedidos,            -- esperado 99.441
  (SELECT COUNT_IF(valido) FROM horizonte.silver.marketplace_pedidos)          AS pedidos_validos,    -- esperado 98.199
  (SELECT COUNT(*) FROM horizonte.silver.marketplace_itens)                    AS itens,              -- esperado 112.650
  (SELECT COUNT_IF(valido AND incluir AND periodo_completo)
     FROM horizonte.silver.marketplace_itens)                                  AS itens_para_gold,
  (SELECT MAX(data_compra) FROM horizonte.silver.marketplace_pedidos
    WHERE periodo_completo)                                                    AS fim_janela;         -- esperado início de set/2026